Recorte de enunciados de PDFs de emestrada — paso a paso

In [1]:
# %pip install pdfplumber pypdf pandas

import re
import os
import shutil
from pathlib import Path
import unicodedata

import pandas as pd
import pdfplumber
from pypdf import PdfReader, PdfWriter, Transformation
from IPython.display import display

## Establecer directorio raiz

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

## Parámetros para medidas de recuadros y pdf

In [3]:
ANCHO_MIN, ALTO_MIN = 450, 20   # tamaño mínimo para considerar un rectángulo como recuadro de enunciado
MARGEN = 2                      # holgura alrededor del borde al recortar
DPI = 200                       # resolución de los PNG
A4 = (595.28, 841.89)
MARGEN_A4 = 40                  # margen de la hoja de salida
SEPARACION = 15                 # hueco entre recuadros apilados

## detectar los recuadros

In [4]:
def detect_rectangles(page):
    vistos = {}
    for r in page.rects:
        if r["width"] > ANCHO_MIN and r["height"] > ALTO_MIN:
            clave = tuple(round(r[k]) for k in ("x0", "top", "x1", "bottom"))
            vistos[clave] = r
    return sorted(vistos.values(), key=lambda r: r["top"])

In [5]:
# PATRON = re.compile(
#     r"""
#     (?P<asig>SOCIALES[ \t]+(?:II|2)|MATEM[AÁ]TICAS[ \t]+(?:II|2)|QU[IÍ]MICA|F[IÍ]SICA)
#     [ \t.,:;-]*
#     (?P<anio>(?:19|20)\d{2})
#     [ \t.,:;-]*
#     (?P<conv>(?:JUNIO|JULIO|SEPTIEMBRE|PONENCIA|RESERVA|MODELO)(?:[ \t]*\d+)?)
#     [ \t.,:;-]*
#     (?P<ej>[^\n\r]*?)
#     [ \t.]*\r?$
#     """,
#     re.IGNORECASE | re.MULTILINE | re.VERBOSE,
# )

In [6]:
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.upper())
    s = "".join(c for c in s if not unicodedata.combining(c))  # quita tildes
    return re.sub(r"\s+", "", s)

## Extraer info del ejercicio

In [7]:
def process_exercise_info(exercise_info):
    exercise_info = exercise_info.lower()

    # valores por defecto si no puede procesarse
    # la información del ejercicio
    (asignatura, year, convocatoria, ejercicio) = ("" for _ in range(4))
    
    asignaturas = {
        "quimica" : "Química",
        "química" : "Química",
        "física" : "Física",
        "fisica" : "Física",
        "sociales ii" : "Mates CCSS",
        "matematicas ii" : "Matemáticas II"
    }

    # extraer asignatura
    for k,v in asignaturas.items():
        if k in exercise_info:
            asignatura = v
            break
        else:
            asignatura = "asignatura_desconocida"

    # extraer año
    year_pattern = r"2\d{3}"
    year = re.findall(year_pattern, exercise_info)[0]

    # extraer convocatoria y ejercicio
    convocatorias = ["junio", "julio", "septiembre",
                     "modelo", "ponencia",
                     "reserva"]

    # convocatorias de Junio, Julio, Septiembre, modelo o ponencia
    for conv in convocatorias[:-2]:
        if conv in exercise_info:
            convocatoria = conv.title()
            convocatoria_position = exercise_info.find(conv)
            ejercicio = (" ".join(exercise_info[convocatoria_position:].split(" ")[1:])
                         .title()
            )
            break

    # convocatoria es una reserva
    if "reserva" in exercise_info:
        convocatoria_position = exercise_info.find("reserva")
        convocatoria = (" ".join(exercise_info[convocatoria_position:]
                                 .split(" ")[:2])
                                 .title()
                                 .replace(".", "")
        )
        ejercicio = (" ".join(
            exercise_info[convocatoria_position:].split(" ")[2:]
            ).title()
        )
    
    return (asignatura, year, convocatoria, ejercicio)   


## Limpieza de texto

In [8]:
def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", texto).strip("_").lower()

#print(slug("RESERVA 1"), "|", slug("SEPTIEMBRE"), "|", slug("2023 - Integrales"))

## Procesamiento pdf

In [39]:
def procesar_pdf(ruta_pdf, output_folder, export_pdf = True, export_png=True, dpi=DPI, one_ex_per_page=False):
    exercises, clips = [], []
    # ruta_pdf
    # 'clasificador_selectividad/source_pdf_files/Física - Temas/campo_eléctrico_magnético/2016 - Campo eléctrico y magnético.pdf'
    path_parts = ruta_pdf.parts
    subject_path = path_parts[-3].replace(" - Temas", "")
    topic_path = path_parts[-2]
    png_folder = output_folder / "png_enunciados" / f"png_{subject_path}_{ruta_pdf.stem}"

    # 1) Detección con pdfplumber
    with pdfplumber.open(ruta_pdf) as pdf:
        # procesamiento de las páginas del pdf
        for n, page in enumerate(pdf.pages):
            # detectar los recuadros
            boxes = detect_rectangles(page)

            # obtener coordenadas del recuadro
            for k, r in enumerate(boxes, start=1):
                x0 = max(r["x0"] - MARGEN, 0)
                x1 = min(r["x1"] + MARGEN, page.width)
                top = max(r["top"] - MARGEN, 0)
                bottom = min(r["bottom"] + MARGEN, page.height)

                # recortar el trozo del recuadro del pdf
                clip = page.crop((x0, top, x1, bottom))

                # extraer el texto del enunciado en
                # el recuadro
                texto_raw = clip.extract_text() or ""

                # separar el enunciado en líneas
                lineas = texto_raw.split("\n")

                # el enunciado son todas las líneas excepto
                # la última (información del ejercicio)
                texto = "\n".join(lineas[:-1])

                # información del ejercicio
                exercise_info = lineas[-1]
                asignatura, anio, conv, ej = process_exercise_info(exercise_info)
                # texto = texto_raw.replace("\n", " ")
                # metadata = PATRON.search(texto_raw)   # buscar con saltos de línea

                # crear nombre de archivo png
                # a partir de la información del ejercicio
                if exercise_info:
                    ej_slug = slug(ej)
                    png_name = f"{asignatura}_{topic_path}_{anio}_{slug(conv)}_{ej_slug}"
                    # ej_slug = slug(re.sub(r"EJERCICIO", "", metadata["ej"], flags=re.IGNORECASE))
                    # nombre = f"{metadata['anio']}_{slug(metadata['conv'])}_{ej_slug}"
                else:
                    png_name = f"{slug(ruta_pdf.stem)}_p{n + 1:02d}"

                # exportar recorte a png
                image = None
                if export_png:
                    png_folder.mkdir(parents=True, exist_ok=True)
                    output_file = png_folder / f"{png_name}.png"
                    i = 2
                    while output_file.exists():
                        output_file = png_folder / f"{png_name}_{i}.png"
                        i += 1
                    clip.to_image(resolution=dpi).save(output_file)
                    image = str(output_file.relative_to(output_folder))

                off_x, off_y = float(page.bbox[0]), float(page.bbox[1])
                clips.append((n, off_x + x0, off_y + page.height - bottom,
                                 off_x + x1, off_y + page.height - top))

                aviso = ""
                if not exercise_info:
                    aviso = "sin info ejercicio"
                elif len(boxes) > 1:
                    aviso = f"{len(boxes)} recuadros en la página"

                # asignatura y tema
                #asignatura = ruta_pdf.parts[-3].replace(" - Temas", "").replace(" ", "").lower()
                tema = ruta_pdf.stem.split(" - ")[1]
                if asignatura == "asignatura_desconocida":
                    asignatura = subject_path

                exercise_info_dict = {
                    "asignatura": asignatura,
                    "asignatura_carpeta" : subject_path,
                    "año" : anio,
                    "convocatoria" : conv,
                    "ejercicio" : ej,
                    "tema": tema,
                    "info_ejercicio" : exercise_info,
                    "enunciado": texto,
                    "archivo": ruta_pdf.name,
                    "imagen": image,
                    "aviso": aviso,
                    #"año": metadata["anio"] if metadata else "",
                    #"convocatoria": metadata["conv"].title() if metadata else "",
                    #"ejercicio": metadata["ej"] if metadata else "",
                }

                # si export_png = False, borra esa clave del exercises
                if not export_png:
                    del exercise_info_dict["imagen"]
                
                exercises.append(exercise_info_dict)              


    if not clips:
        return pd.DataFrame(), "ningún recuadro detectado"
    
    # combinar todos los enunciados en un pdf
    if export_pdf:
        # 2) Composición vectorial con pypdf
        # comprobar si existe el directorio de salida y crearlo si no
        (output_folder / "pdf_enunciados").mkdir(parents=True, exist_ok = True)
        output_pdf_filename = output_folder / "pdf_enunciados" / f"{subject_path}_{ruta_pdf.stem}_enunciados.pdf"
        writer = PdfWriter()
        hoja, y, num_hoja = None, None, 0
        already_processed = set()
        reader = PdfReader(ruta_pdf)

        for ex, (n, x0, y0, x1, y1) in zip(exercises, clips):
            # si la página ya se usó (varios recuadros), se relee para no pisar el recorte anterior
            src = (PdfReader(ruta_pdf) if n in already_processed else reader).pages[n]
            already_processed.add(n)
            src.mediabox.lower_left = src.cropbox.lower_left = (x0, y0)
            src.mediabox.upper_right = src.cropbox.upper_right = (x1, y1)
            w, h = x1 - x0, y1 - y0

            if one_ex_per_page:
                hoja = writer.add_blank_page(w, h)
                hoja.merge_transformed_page(src, Transformation().translate(-x0, -y0))
                num_hoja += 1
            else:
                escala = min(1.0, (A4[0] - 2 * MARGEN_A4) / w)
                w, h = w * escala, h * escala
                if hoja is None or y - h < MARGEN_A4:
                    hoja = writer.add_blank_page(*A4)
                    y = A4[1] - MARGEN_A4
                    num_hoja += 1
                tx, ty = (A4[0] - w) / 2, y - h
                t = Transformation().translate(-x0, -y0).scale(escala, escala).translate(tx, ty)
                hoja.merge_transformed_page(src, t)
                y -= h + SEPARACION

            # fila["pdf_salida"] = salida.name
            # fila["pagina_salida"] = num_hoja

        # guardar en pdf los enunciados
        with open(output_pdf_filename, "wb") as f:
            writer.write(f)

    return pd.DataFrame(exercises), ""

## Procesar carpeta

In [10]:
def process_folder(folder, export_pdf = True, export_png = True, files_to_process_limit=None):

    # '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_gravitatorio'
    source_folder = Path(folder)
    output_folder_name = "/".join(source_folder.parts[-2:])
    output_folder = root / f"solo_enunciados_ejercicios/{output_folder_name}"
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Procesando carpeta: {output_folder_name}")

    # obtener rutas de los archivos pdf de la carpeta
    pdf_files = sorted(p for p in source_folder.rglob("*.pdf") if not p.stem.endswith("_enunciados"))
    
    # hay límite de archivos por procesar
    if files_to_process_limit:
        pdf_files = pdf_files[:files_to_process_limit]

    print(f"{len(pdf_files)} PDFs a procesar\n")

    dataframes, issues = [], []
    for index, file_path in enumerate(pdf_files, start=1):
        try:
            df, aviso = procesar_pdf(file_path, output_folder, export_pdf = export_pdf, export_png = export_png)
        except Exception as e:
            subject = file_path.parts[-3].replace(" - Temas", "")
            issues.append({"issue": f"ERROR: {e}",
                           "subject" : subject,
                           "pdf file": str(file_path.name)})
            print(f"[{index}/{len(pdf_files)}] ERROR  ({subject}) {file_path.name}: {e}")
            continue

        dataframes.append(df)

        if aviso:
            issues.append({"issue": aviso,
                           "pdf file": str(file_path)})
            
            print(f"[{index}/{len(pdf_files)}] AVISO  {file_path.name}: {aviso}")
        else:
            print(f"[{index}/{len(pdf_files)}] OK     {file_path.name}: {len(df)} ejercicios procesados")

    # Guardar resultados en CSV
    enunciados_df = pd.DataFrame()
    for d in dataframes:
        enunciados_df = pd.concat([enunciados_df, d])
            
    enunciados_df.to_csv(output_folder / f"{output_folder.name}.csv", index=False)

    if issues:
        issues_df = pd.DataFrame(issues)
        issues_df.to_csv(output_folder / f"{output_folder.name}_issues.csv",
                            index=False, encoding="utf-8")

    print("*" * 40)
    print(f"\nProcesamiento completado. Resultados guardados en: {output_folder}\n\n")

## Procesar carpetas

Normalizar texto de nombres de temas y carpetas

In [11]:
def normalize_text(texto):
    descompuesto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))

Obtener las rutas de los archivos

In [12]:
os.chdir(f"{root}/source_pdf_files")

In [13]:
carpetas_asignaturas = {d.name:[] for d in Path(".").iterdir() if d.is_dir()}

In [14]:
for asignatura in carpetas_asignaturas:
    carpetas_asignaturas[asignatura] = [d.name for d in Path(f"./{asignatura}").iterdir() if d.is_dir()]

In [15]:
folder_paths = []
for asignatura, carpetas in carpetas_asignaturas.items():
    for carpeta in carpetas:
        folder_paths.append(f"{root}/source_pdf_files/{asignatura}/{carpeta}")

In [16]:
# folder_paths

## Procesar todos los archivos

In [17]:
# process_folder(folder_paths[0], files_to_process_limit=None)

In [18]:
for folder in folder_paths:
    folder_path = Path(folder)
    # Limitar a 5 archivos por carpeta para pruebas
    process_folder(folder, export_pdf = True, export_png = False, files_to_process_limit=None)

Procesando carpeta: Física - Temas/campo_gravitatorio
11 PDFs a procesar

[1/11] OK     2016 - Campo gravitatorio.pdf: 10 ejercicios procesados
[2/11] OK     2017 - Campo gravitatorio.pdf: 12 ejercicios procesados
[3/11] OK     2018 - Campo gravitatorio.pdf: 12 ejercicios procesados
[4/11] OK     2019 - Campo gravitatorio.pdf: 12 ejercicios procesados
[5/11] OK     2020 - Campo gravitatorio.pdf: 12 ejercicios procesados
[6/11] OK     2021 - Campo gravitatorio.pdf: 12 ejercicios procesados
[7/11] OK     2022 - Campo gravitatorio.pdf: 12 ejercicios procesados
[8/11] OK     2023 - Campo gravitatorio.pdf: 12 ejercicios procesados
[9/11] OK     2024 - Campo gravitatorio.pdf: 12 ejercicios procesados
[10/11] OK     2025 - Campo gravitatorio.pdf: 15 ejercicios procesados
[11/11] OK     2026 - Campo gravitatorio.pdf: 2 ejercicios procesados
****************************************

Procesamiento completado. Resultados guardados en: /home/daniel/code/clasificador_selectividad/solo_enunciados_ej

## Mover archivos .csv

In [25]:
os.chdir(root)
Path("csv_enunciados").mkdir(exist_ok=True)
os.chdir("solo_enunciados_ejercicios")

In [27]:
output_csv_path = root / "csv_enunciados"
for folder in Path(".").iterdir():
    files = folder.rglob("*.csv")
    subject = folder.name.replace(" - Temas", "").replace(" ", "_").lower()
        
    for csv_file in files:
        new_filename = f"{subject}_{csv_file.stem}.csv"
        #print(new_filename)
        shutil.move(csv_file, output_csv_path / new_filename)

Combinar todos los archivos csv

In [42]:
os.chdir(root / "csv_enunciados")
exercises_df = pd.DataFrame()
issues_df = pd.DataFrame()

for file in Path(".").iterdir():
    if file.suffix == ".csv":
        if "issue" in file.name:
            df = pd.read_csv(file, encoding="utf-8")
            issues_df = pd.concat([issues_df, df])
        else:
            df = pd.read_csv(file, encoding="utf-8")
            exercises_df = pd.concat([exercises_df, df])
    
issues_df.to_csv("issues.csv", index = False)

# sort exercises
exercises_df = exercises_df.sort_values(by=["asignatura", "año", "convocatoria", "ejercicio"])
exercises_df.to_csv("enunciados_ejercicios.csv", index = False)

Comprobar que `enunciados_ejercicios.csv` existe

In [43]:
os.chdir(root / "csv_enunciados")
enunciados_path = root / "csv_enunciados" / "enunciados_ejercicios.csv"
if enunciados_path.exists() and os.path.getsize(enunciados_path) > 0:
    files_to_remove = [f for f in Path(".").iterdir()
                       if f.suffix == ".csv" and
                       f.name not in ["enunciados_ejercicios.csv", "issues.csv"]
    ]
    for f in files_to_remove:
        f.unlink()

In [29]:
issues_df

,issue,subject,pdf file
0,ERROR: list index out of range,Química,2021 - Enlace Químico.pdf
1,ningún recuadro detectado,NaN,/home/daniel/code/clasificador_selectividad/so...
0,ningún recuadro detectado,NaN,/home/daniel/code/clasificador_selectividad/so...
1,ningún recuadro detectado,NaN,/home/daniel/code/clasificador_selectividad/so...
0,ERROR: list index out of range,Mates II,Tipos ejercicios matrices.pdf
0,ningún recuadro detectado,NaN,/home/daniel/code/clasificador_selectividad/so...
0,ERROR: list index out of range,Química,2008 - Reactividad Orgánica.pdf
0,ERROR: list index out of range,Mates CCSS,2016 - Funciones CCSS.pdf
0,ningún recuadro detectado,NaN,/home/daniel/code/clasificador_selectividad/so...


In [44]:
todos_df = pd.read_csv("enunciados_ejercicios.csv")
todos_df.sample(10)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2095,Mates II,Mates II,2008,Reserva 2,Ejercicio 2. Opción B.,Integrales,MATEMÁTICAS II. 2008. RESERVA 2. EJERCICIO 2. ...,1\nCalcula xln(x1)dx .\n0,2008 - Integrales.pdf,NaN,NaN
3405,Química,Química,2006,Junio,Ejercicio 4. Opción A,Ácido Base,QUÍMICA. 2006. JUNIO. EJERCICIO 4. OPCIÓN A,"Utilizando la teoría de Brönsted-Lowry, justif...",2006 - Ácido Base.pdf,NaN,NaN
4005,Química,Química,2014,Reserva 2,Ejercicio 6. Opción A,Ácido Base,QUÍMICA. 2014. RESERVA 2. EJERCICIO 6. OPCIÓN A,Una disolución acuosa 102M de ácido benzoico ...,2014 - Ácido Base.pdf,NaN,NaN
973,Mates CCSS,Mates CCSS,2011,Reserva 1,Ejercicio 1. Opcion B,Matrices Y Determinantes,SOCIALES II. 2011 RESERVA 1. EJERCICIO 1. OPCI...,0 3 1  2 3 1\na) Dadas las matrices M ...,2011 - Matrices Y Determinantes.pdf,NaN,NaN
581,Mates CCSS,Mates CCSS,2003,Reserva 1,Ejercicio 3. Parte I. Opción A,Probabilidad,S OCIALES II. 2003 RESERVA 1. EJERCICIO 3. PAR...,E n una residencia hay 212 ancianos de los que...,2003 - Probabilidad.pdf,NaN,NaN
1952,Mates II,Mates II,2005,Reserva 2,Ejercicio 2. Opción B.,Integrales,MATEMÁTICAS II. 2005. RESERVA 2. EJERCICIO 2. ...,"0\nCalcula  Ln(2 x)dx, siendo Ln la función ...",2005 - Integrales.pdf,NaN,NaN
135,Física,Física,2018,Septiembre,Ejercicio 4. Opción B,Física cuántica y nuclear,FISICA. 2018. SEPTIEMBRE. EJERCICIO 4. OPCIÓN B,"g = 9,8 m s-2\na) Se ilumina la superficie de ...",2018 - Física cuántica y nuclear.pdf,NaN,NaN
4322,Química,Química,2019,Junio,Ejercicio 6. Opción B,Reacciones Redox,QUIMICA. 2019. JUNIO. EJERCICIO 6. OPCIÓN B,El bromuro de potasio reacciona con ácido sulf...,2019 - Reacciones Redox.pdf,NaN,NaN
1128,Mates CCSS,Mates CCSS,2014,Reserva 2,Ejercicio 2. Opción B,Funciones CCSS,SOCIALES II. 2014. RESERVA 2. EJERCICIO 2. OPC...,"x2 ax5 si x0\nSea la función f , definida...",2014 - Funciones CCSS.pdf,NaN,NaN
2944,Mates II,Mates II,2025,Reserva 3,Ejercicio 2,Funciones,MATEMÁTICAS II. 2025. RESERVA 3. EJERCICIO 2,"Considera la función f : ( − 1 , 1 ) →\n1\ndef...",2025 - Funciones.pdf,NaN,NaN


In [45]:
todos_df.head()

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
0,Física,Física,2016,Junio,Ejercicio 1. Opción A,Campo eléctrico y magnético,FISICA. 2016. JUNIO. EJERCICIO 1. OPCIÓN A,a) Campo eléctrico creado por una carga puntua...,2016 - Campo eléctrico y magnético.pdf,NaN,NaN
1,Física,Física,2016,Junio,Ejercicio 1. Opción B,Campo gravitatorio,FISICA. 2016. JUNIO. EJERCICIO 1. OPCIÓN B,a) Defina velocidad de escape de un planeta y ...,2016 - Campo gravitatorio.pdf,NaN,NaN
2,Física,Física,2016,Junio,Ejercicio 2. Opción A,Ondas,FISICA. 2016. JUNIO. EJERCICIO 2. OPCIÓN A,a) Explique las características cinemáticas de...,2016 - Ondas.pdf,NaN,NaN
3,Física,Física,2016,Junio,Ejercicio 2. Opción B,Física cuántica y nuclear,FISICA. 2016. JUNIO. EJERCICIO 2. OPCIÓN B,a) Teoría de Einstein del efecto fotoeléctrico...,2016 - Física cuántica y nuclear.pdf,NaN,NaN
4,Física,Física,2016,Junio,Ejercicio 3. Opción A,Campo gravitatorio,FISICA. 2016. JUNIO. EJERCICIO 3. OPCIÓN A,Dos partículas de masas m 3kg y m  5kg se en...,2016 - Campo gravitatorio.pdf,NaN,NaN


In [ ]:
todos_df.asignatura.value_counts()

asignatura
Química       1808
Mates II      1267
Mates CCSS    1213
Física         486
Name: count, dtype: int64

In [32]:
todos_df[todos_df.asignatura == "asignatura_desconocida"]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso


In [33]:
todos_df[todos_df.enunciado == ""]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso


In [34]:
todos_df.convocatoria.unique()

<StringArray>
[             'Junio',          'Reserva 2',          'Reserva 3',
          'Reserva 4',         'Septiembre',          'Reserva 1',
              'Julio',                  nan,             'Modelo',
 'Reserva1 Ejercicio']
Length: 10, dtype: str

In [35]:
todos_df.loc[0, :]

asignatura                                                       Física
asignatura_carpeta                                               Física
año                                                                2016
convocatoria                                                      Junio
ejercicio                                         Ejercicio 2. Opción B
tema                                          Física cuántica y nuclear
info_ejercicio               FISICA. 2016. JUNIO. EJERCICIO 2. OPCIÓN B
enunciado             a) Teoría de Einstein del efecto fotoeléctrico...
archivo                            2016 - Física cuántica y nuclear.pdf
imagen                                                              NaN
aviso                                                               NaN
Name: 0, dtype: object

In [36]:
todos_df.columns

Index(['asignatura', 'asignatura_carpeta', 'año', 'convocatoria', 'ejercicio',
       'tema', 'info_ejercicio', 'enunciado', 'archivo', 'imagen', 'aviso'],
      dtype='str')